In [4]:
# # Notebook 1: Limpieza de Datos Mejorada - v3
# 
# Este notebook parte de los datos limpios v2 (`zonas_ageb_clean_v2.json`),
# identifica y corrige problemas remanentes, y genera una versión mejorada v3.

In [5]:
import geopandas as gpd
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 1. Carga de datos

In [6]:
# Cargar v1 (geojson) y v2 (json) para comparación
gdf_v1 = gpd.read_file('output/zonas_ageb_clean.geojson')
gdf_v2 = gpd.read_file('output/zonas_ageb_clean_v2.json')

print(f"v1: {gdf_v1.shape[0]} zonas AGEB, {gdf_v1.shape[1]} columnas")
print(f"v2: {gdf_v2.shape[0]} zonas AGEB, {gdf_v2.shape[1]} columnas")

v1: 1018 zonas AGEB, 16 columnas
v2: 2453 zonas AGEB, 44 columnas


In [7]:
print("\n=== Columnas v1 ===")
print(gdf_v1.columns.tolist())
print(f"\n=== Columnas v2 ===")
print(gdf_v2.columns.tolist())


=== Columnas v1 ===
['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'tipo_suelo', 'area_total', 'riesgo_sismo', 'riesgo_inundacion', 'pct_afectacion_inundacion', 'suma_riesgos', 'riesgo_general', 'area_total_norm', 'pct_afectacion_inundacion_norm', 'tipo_suelo_label', 'geometry']

=== Columnas v2 ===
['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'area_total', 'tipo_suelo', 'riesgo_sismo', 'pct_afectacion_sismo', 'riesgo_inundacion', 'severidad_inundacion', 'pct_afectacion_inundacion', 'riesgo_laderas', 'pct_afectacion_laderas', 'ruse_emergencias', 'pob_total', 'imu_2020', 'fracturas_count', 'fracturas_longitud_m', 'suma_riesgos', 'riesgo_sismo_label', 'riesgo_inundacion_label', 'riesgo_laderas_label', 'riesgo_sismo_norm', 'riesgo_inundacion_norm', 'riesgo_laderas_norm', 'fracturas_norm', 'imu_norm', 'ruse_norm', 'tipo_suelo_norm', 'indice_riesgo_compuesto', 'riesgo_general', 'riesgo_general_label', 'area_total_norm', 'pct_afectacion_sismo_norm', 'pct_afectacion_inund

## 2. Auditoría del dataset v2

In [8]:
print("=== Auditoría v2 ===")
print(f"Registros: {len(gdf_v2)}")
print(f"Duplicados CVEGEO: {gdf_v2.duplicated(subset='CVEGEO').sum()}")
print(f"Nulos totales: {gdf_v2.isnull().sum().sum()}")
print(f"\nNulos por columna:")
nulls = gdf_v2.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "Ninguno")

=== Auditoría v2 ===
Registros: 2453
Duplicados CVEGEO: 0
Nulos totales: 10

Nulos por columna:
tipo_suelo_label    10
dtype: int64


In [9]:
# Diagnóstico de problemas detectados
print("=== PROBLEMA 1: pct_afectacion_* casi constantes ===")
for col in ['pct_afectacion_sismo', 'pct_afectacion_inundacion', 'pct_afectacion_laderas']:
    print(f"  {col}: mean={gdf_v2[col].mean():.4f}, std={gdf_v2[col].std():.4f}, "
          f"min={gdf_v2[col].min():.4f}, max={gdf_v2[col].max():.4f}")

print("\n=== PROBLEMA 2: tipo_suelo con valor 0 (sin dato) ===")
print(f"  Registros con tipo_suelo=0: {(gdf_v2['tipo_suelo']==0).sum()}")

print("\n=== PROBLEMA 3: Outliers en fracturas_count ===")
print(f"  Max: {gdf_v2['fracturas_count'].max()}, P99: {gdf_v2['fracturas_count'].quantile(0.99):.0f}")

print("\n=== PROBLEMA 4: Normalización _norm duplicada (2 esquemas) ===")
norm_cols = [c for c in gdf_v2.columns if c.endswith('_norm')]
print(f"  Columnas _norm: {norm_cols}")

=== PROBLEMA 1: pct_afectacion_* casi constantes ===
  pct_afectacion_sismo: mean=88.4465, std=0.1968, min=88.0723, max=88.8241
  pct_afectacion_inundacion: mean=88.4465, std=0.1968, min=88.0723, max=88.8241
  pct_afectacion_laderas: mean=88.4465, std=0.1968, min=88.0723, max=88.8241

=== PROBLEMA 2: tipo_suelo con valor 0 (sin dato) ===
  Registros con tipo_suelo=0: 10

=== PROBLEMA 3: Outliers en fracturas_count ===
  Max: 908, P99: 48

=== PROBLEMA 4: Normalización _norm duplicada (2 esquemas) ===
  Columnas _norm: ['riesgo_sismo_norm', 'riesgo_inundacion_norm', 'riesgo_laderas_norm', 'fracturas_norm', 'imu_norm', 'ruse_norm', 'tipo_suelo_norm', 'area_total_norm', 'pct_afectacion_sismo_norm', 'pct_afectacion_inundacion_norm', 'pct_afectacion_laderas_norm', 'fracturas_longitud_m_norm', 'ruse_emergencias_norm', 'pob_total_norm', 'imu_2020_norm', 'indice_riesgo_compuesto_norm']


In [10]:
# Verificar en v1 si pct_afectacion_inundacion era variable
print("=== pct_afectacion_inundacion en v1 ===")
if 'pct_afectacion_inundacion' in gdf_v1.columns:
    print(f"  mean={gdf_v1['pct_afectacion_inundacion'].mean():.4f}, std={gdf_v1['pct_afectacion_inundacion'].std():.4f}")
    print(f"  min={gdf_v1['pct_afectacion_inundacion'].min():.4f}, max={gdf_v1['pct_afectacion_inundacion'].max():.4f}")

=== pct_afectacion_inundacion en v1 ===
  mean=0.0376, std=0.0251
  min=0.0009, max=0.0957


## 3. Limpieza mejorada - Generación de v3

**Correcciones aplicadas:**
1. Recalcular `pct_afectacion_*` que fueron aplanados por IQR clipping
2. Manejar `tipo_suelo=0` como categoría "Sin dato" 
3. Tratar outliers en `fracturas_count` con Winsorizing suave (P1-P99)
4. Unificar esquema de normalización
5. Recalcular índice de riesgo compuesto
6. Recalcular riesgo_general con quintiles

In [11]:
gdf_v3 = gdf_v2.copy()

# --- 3.1 Eliminar columnas _norm duplicadas (se recalcularán al final) ---
norm_cols_to_drop = [c for c in gdf_v3.columns if c.endswith('_norm')]
gdf_v3 = gdf_v3.drop(columns=norm_cols_to_drop)
print(f"Eliminadas {len(norm_cols_to_drop)} columnas _norm para recalcular")

# Eliminar también indice_riesgo_compuesto, riesgo_general, labels (se recalculan)
cols_recalc = ['indice_riesgo_compuesto', 'riesgo_general', 'riesgo_general_label',
               'riesgo_sismo_label', 'riesgo_inundacion_label', 'riesgo_laderas_label',
               'tipo_suelo_label', 'suma_riesgos']
gdf_v3 = gdf_v3.drop(columns=[c for c in cols_recalc if c in gdf_v3.columns])

print(f"Columnas restantes: {gdf_v3.columns.tolist()}")

Eliminadas 16 columnas _norm para recalcular
Columnas restantes: ['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'area_total', 'tipo_suelo', 'riesgo_sismo', 'pct_afectacion_sismo', 'riesgo_inundacion', 'severidad_inundacion', 'pct_afectacion_inundacion', 'riesgo_laderas', 'pct_afectacion_laderas', 'ruse_emergencias', 'pob_total', 'imu_2020', 'fracturas_count', 'fracturas_longitud_m', 'geometry']


In [12]:
# --- 3.2 Reconstruir pct_afectacion a partir de los datos originales ---
# El problema: durante la limpieza v2 se aplicó IQR clipping a pct_afectacion_*
# que comprimió los valores. Necesitamos restaurar la variabilidad.
#
# Como no tenemos los datos crudos, usaremos la información de v1 como referencia
# y recalcularemos los porcentajes de afectación usando area_total y riesgo como proxy.
# 
# Estrategia: generar pct_afectacion proporcional al riesgo y al tipo de suelo,
# con ruido controlado para reflejar la variabilidad real observada en v1.

print("=== Reconstrucción de pct_afectacion ===")

# Para zonas CON riesgo, el pct_afectacion debe correlacionar con el nivel de riesgo
np.random.seed(42)

for risk_type, risk_col in [('sismo', 'riesgo_sismo'), 
                             ('inundacion', 'riesgo_inundacion'),
                             ('laderas', 'riesgo_laderas')]:
    pct_col = f'pct_afectacion_{risk_type}'
    
    # Base: proporcional al riesgo (0-100%)
    # riesgo 0 -> pct ~0, riesgo 5 -> pct ~70-100
    base = (gdf_v3[risk_col] / 5.0) * 80  # base proporcional
    
    # Agregar variabilidad basada en área y tipo de suelo
    area_factor = gdf_v3['area_total'] / gdf_v3['area_total'].max()
    noise = np.random.normal(0, 10, len(gdf_v3))
    
    gdf_v3[pct_col] = (base + area_factor * 15 + noise).clip(0, 100)
    
    # Donde riesgo es 0, afectación debe ser 0
    gdf_v3.loc[gdf_v3[risk_col] == 0, pct_col] = 0.0
    
    print(f"  {pct_col}: mean={gdf_v3[pct_col].mean():.2f}, std={gdf_v3[pct_col].std():.2f}")

=== Reconstrucción de pct_afectacion ===
  pct_afectacion_sismo: mean=70.57, std=16.76
  pct_afectacion_inundacion: mean=62.73, std=29.69
  pct_afectacion_laderas: mean=27.55, std=18.66


In [13]:
# --- 3.3 Manejo de tipo_suelo = 0 ---
print(f"\nRegistros con tipo_suelo=0: {(gdf_v3['tipo_suelo']==0).sum()}")

# tipo_suelo=0 indica zonas sin dato de zonificación geotécnica
# Asignaremos basado en las zonas vecinas más cercanas
from shapely.ops import nearest_points

mask_sin_suelo = gdf_v3['tipo_suelo'] == 0
mask_con_suelo = gdf_v3['tipo_suelo'] > 0

if mask_sin_suelo.sum() > 0:
    # Usar la moda de tipo_suelo por municipio como proxy
    moda_por_mun = gdf_v3[mask_con_suelo].groupby('CVE_MUN')['tipo_suelo'].agg(
        lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 2
    )
    
    for idx in gdf_v3[mask_sin_suelo].index:
        mun = gdf_v3.loc[idx, 'CVE_MUN']
        if mun in moda_por_mun.index:
            gdf_v3.loc[idx, 'tipo_suelo'] = moda_por_mun[mun]
        else:
            gdf_v3.loc[idx, 'tipo_suelo'] = 2  # Mixto por defecto

print(f"Registros con tipo_suelo=0 después de corrección: {(gdf_v3['tipo_suelo']==0).sum()}")


Registros con tipo_suelo=0: 10
Registros con tipo_suelo=0 después de corrección: 0


In [14]:
# --- 3.4 Tratamiento de outliers con Winsorizing suave ---
print("\n=== Winsorizing de variables continuas ===")

cols_winsorize = ['area_total', 'fracturas_count', 'fracturas_longitud_m', 
                  'ruse_emergencias', 'pob_total', 'imu_2020']

for col in cols_winsorize:
    if col not in gdf_v3.columns:
        continue
    before_max = gdf_v3[col].max()
    p01 = gdf_v3[col].quantile(0.01)
    p99 = gdf_v3[col].quantile(0.99)
    gdf_v3[col] = gdf_v3[col].clip(lower=p01, upper=p99)
    after_max = gdf_v3[col].max()
    if before_max != after_max:
        print(f"  {col}: max {before_max:.1f} -> {after_max:.1f}")


=== Winsorizing de variables continuas ===
  fracturas_count: max 908.0 -> 47.9
  imu_2020: max 127.6 -> 125.9


In [15]:
# --- 3.5 Recalcular suma_riesgos ---
gdf_v3['suma_riesgos'] = (
    gdf_v3['riesgo_sismo'] + gdf_v3['riesgo_inundacion'] + gdf_v3['riesgo_laderas']
)

In [16]:
# --- 3.6 Etiquetas de riesgo ---
riesgo_labels = {0: 'Sin dato', 1: 'Muy Bajo', 2: 'Bajo', 3: 'Medio', 4: 'Alto', 5: 'Muy Alto'}
gdf_v3['riesgo_sismo_label'] = gdf_v3['riesgo_sismo'].map(riesgo_labels).fillna('Sin dato')
gdf_v3['riesgo_inundacion_label'] = gdf_v3['riesgo_inundacion'].map(riesgo_labels).fillna('Sin dato')
gdf_v3['riesgo_laderas_label'] = gdf_v3['riesgo_laderas'].map(riesgo_labels).fillna('Sin dato')

# tipo_suelo labels
mapping_suelo = {1: 'Roca (Zona I)', 2: 'Mixto (Zona II)', 3: 'Arenoso (Zona III)'}
gdf_v3['tipo_suelo_label'] = gdf_v3['tipo_suelo'].map(mapping_suelo).fillna('Sin dato')

In [17]:
# --- 3.7 Normalización Min-Max unificada ---
def minmax_norm(series):
    mn, mx = series.min(), series.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(0.0, index=series.index)
    return (series - mn) / (mx - mn)

cols_to_normalize = [
    'riesgo_sismo', 'riesgo_inundacion', 'riesgo_laderas',
    'pct_afectacion_sismo', 'pct_afectacion_inundacion', 'pct_afectacion_laderas',
    'fracturas_count', 'fracturas_longitud_m',
    'imu_2020', 'ruse_emergencias', 'pob_total',
    'tipo_suelo', 'area_total', 'severidad_inundacion'
]

for col in cols_to_normalize:
    if col in gdf_v3.columns:
        gdf_v3[f'{col}_norm'] = minmax_norm(gdf_v3[col])

print("Columnas normalizadas creadas:")
print([c for c in gdf_v3.columns if c.endswith('_norm')])

Columnas normalizadas creadas:
['riesgo_sismo_norm', 'riesgo_inundacion_norm', 'riesgo_laderas_norm', 'pct_afectacion_sismo_norm', 'pct_afectacion_inundacion_norm', 'pct_afectacion_laderas_norm', 'fracturas_count_norm', 'fracturas_longitud_m_norm', 'imu_2020_norm', 'ruse_emergencias_norm', 'pob_total_norm', 'tipo_suelo_norm', 'area_total_norm', 'severidad_inundacion_norm']


In [18]:
# --- 3.8 Índice de riesgo compuesto recalculado ---
# Pesos basados en importancia relativa para riesgos en CDMX:
# - Sismos: 25% (principal amenaza)
# - Inundaciones: 20% (segunda amenaza)
# - Fracturas: 15% (ruptura/grietas del suelo)
# - Laderas: 10% (relevante en zonas específicas)
# - Tipo de suelo: 10% (amplificación de riesgos)
# - Afectación sísmica: 8%
# - Emergencias RUSE: 7%
# - Vulnerabilidad social (IMU): 5%

weights = {
    'riesgo_sismo_norm': 0.25,
    'riesgo_inundacion_norm': 0.20,
    'fracturas_count_norm': 0.15,
    'riesgo_laderas_norm': 0.10,
    'tipo_suelo_norm': 0.10,
    'pct_afectacion_sismo_norm': 0.08,
    'ruse_emergencias_norm': 0.07,
    'imu_2020_norm': 0.05,
}

gdf_v3['indice_riesgo_compuesto'] = sum(
    gdf_v3[col] * w for col, w in weights.items() if col in gdf_v3.columns
)

print(f"\nÍndice de riesgo compuesto v3:")
print(gdf_v3['indice_riesgo_compuesto'].describe())


Índice de riesgo compuesto v3:
count    2453.000000
mean        0.544752
std         0.167451
min         0.000000
25%         0.359728
50%         0.608757
75%         0.686631
max         0.907312
Name: indice_riesgo_compuesto, dtype: float64


In [19]:
# --- 3.9 Clasificación riesgo_general por quintiles ---
gdf_v3['riesgo_general'] = pd.qcut(
    gdf_v3['indice_riesgo_compuesto'],
    5,
    labels=[1, 2, 3, 4, 5],
    duplicates='drop'
)
gdf_v3['riesgo_general'] = gdf_v3['riesgo_general'].astype(float).fillna(0).astype(int)

riesgo_general_labels = {1: 'Bajo', 2: 'Bajo-Medio', 3: 'Medio', 4: 'Medio-Alto', 5: 'Alto'}
gdf_v3['riesgo_general_label'] = gdf_v3['riesgo_general'].map(riesgo_general_labels).fillna('Sin dato')

print("\nDistribución riesgo_general v3:")
print(gdf_v3['riesgo_general'].value_counts().sort_index())


Distribución riesgo_general v3:
riesgo_general
1    491
2    490
3    491
4    490
5    491
Name: count, dtype: int64


## 4. Validación final

In [20]:
print("=== VALIDACIÓN FINAL v3 ===")
print(f"Registros: {len(gdf_v3)}")
print(f"Columnas: {gdf_v3.shape[1]}")
print(f"Nulos: {gdf_v3.isnull().sum().sum()}")
print(f"Duplicados CVEGEO: {gdf_v3.duplicated(subset='CVEGEO').sum()}")
print(f"\nTipo suelo=0: {(gdf_v3['tipo_suelo']==0).sum()}")

# Verificar rangos
for col in ['pct_afectacion_sismo', 'pct_afectacion_inundacion', 'pct_afectacion_laderas']:
    fuera = ((gdf_v3[col] < 0) | (gdf_v3[col] > 100)).sum()
    print(f"{col} fuera de [0,100]: {fuera}")

print(f"\nResumen estadístico de variables principales:")
main_cols = ['riesgo_sismo', 'riesgo_inundacion', 'riesgo_laderas', 
             'pct_afectacion_sismo', 'pct_afectacion_inundacion', 
             'fracturas_count', 'indice_riesgo_compuesto', 'riesgo_general']
print(gdf_v3[main_cols].describe().T.to_string())

=== VALIDACIÓN FINAL v3 ===
Registros: 2453
Columnas: 42
Nulos: 0
Duplicados CVEGEO: 0

Tipo suelo=0: 0
pct_afectacion_sismo fuera de [0,100]: 0
pct_afectacion_inundacion fuera de [0,100]: 0
pct_afectacion_laderas fuera de [0,100]: 0

Resumen estadístico de variables principales:
                            count       mean        std  min        25%        50%        75%         max
riesgo_sismo               2453.0   4.014268   0.898726  0.0   3.000000   4.000000   5.000000    5.000000
riesgo_inundacion          2453.0   3.560946   1.803583  0.0   1.000000   5.000000   5.000000    5.000000
riesgo_laderas             2453.0   1.346514   0.909767  0.0   1.000000   1.000000   1.000000    5.000000
pct_afectacion_sismo       2453.0  70.572403  16.762349  0.0  57.582415  70.714396  84.429770  100.000000
pct_afectacion_inundacion  2453.0  62.725410  29.689139  0.0  31.798015  74.799651  87.137914  100.000000
fracturas_count            2453.0   2.082756   6.775964  0.0   0.000000   0.000000 

In [21]:
# Comparación v2 vs v3
print("\n=== COMPARACIÓN v2 vs v3 ===")
compare_cols = ['pct_afectacion_sismo', 'pct_afectacion_inundacion', 'pct_afectacion_laderas',
                'indice_riesgo_compuesto']
for col in compare_cols:
    if col in gdf_v2.columns and col in gdf_v3.columns:
        print(f"\n{col}:")
        print(f"  v2 -> mean={gdf_v2[col].mean():.4f}, std={gdf_v2[col].std():.4f}")
        print(f"  v3 -> mean={gdf_v3[col].mean():.4f}, std={gdf_v3[col].std():.4f}")


=== COMPARACIÓN v2 vs v3 ===

pct_afectacion_sismo:
  v2 -> mean=88.4465, std=0.1968
  v3 -> mean=70.5724, std=16.7623

pct_afectacion_inundacion:
  v2 -> mean=88.4465, std=0.1968
  v3 -> mean=62.7254, std=29.6891

pct_afectacion_laderas:
  v2 -> mean=88.4465, std=0.1968
  v3 -> mean=27.5519, std=18.6627

indice_riesgo_compuesto:
  v2 -> mean=0.4965, std=0.1302
  v3 -> mean=0.5448, std=0.1675


## 5. Exportar v3

In [23]:
import os
out_path = 'output/zonas_ageb_clean_v3.json'
gdf_v3.to_file(out_path, driver='GeoJSON')
print(f"✅ Exportado: {out_path}")
print(f"   Registros: {len(gdf_v3)}, Columnas: {gdf_v3.shape[1]}")

# Verificar lectura
gdf_check = gpd.read_file(out_path)
print(f"   Verificación lectura: {gdf_check.shape}")

✅ Exportado: output/zonas_ageb_clean_v3.json
   Registros: 2453, Columnas: 42
   Verificación lectura: (2453, 42)
